# **Phase_1:** Finding Problematic Files

In [ ]:
import glob
import json
import re
import os
from datetime import datetime
from collections import defaultdict


def extract_key(fname: str) -> int:
    """Extract trailing numeric ID from filename."""
    m = re.search(r'_(\d+)', fname)
    return int(m.group(1)) if m else 0


def validate_json_caption_file(json_path):
    """
    Validate a JSON caption file:
    Returns (ok: bool, error: str).
    """
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            rawr = f.read()
        rawr = re.sub(r"^```(?:json)?\s*", "", rawr)
        rawr = re.sub(r"\s*```$", "", rawr)
        data = json.loads(rawr)
    except Exception as e:
        return False, f"JSON parse error: {e}"

    if not isinstance(data, list):
        return False, "Not a list of captions"

    LANG_FIELDS = {"urdu", "punjabi", "pashto"}


    
    try:
        for i, entry in enumerate(data, start=1):  # line index = caption number
            for field in ["start_time", "end_time", LANG_FIELDS, "english"]:
                if field not in entry:
                    raise ValueError(f"Entry {i} missing field: {field}")

            for tfield in ["start_time", "end_time"]:
                tval = entry[tfield]
                try:
                    datetime.strptime(tval, "%H:%M:%S.%f")
                except ValueError:
                    try:
                        datetime.strptime(tval, "%H:%M:%S")
                    except Exception as e:
                        raise ValueError(f"Entry {i} field '{tfield}' invalid: {tval} ({e})")

    except Exception as e:
        return False, str(e)

    return True, "OK"


def validate_transcripts_only(parent_dir, ver="v0.0"):
    """
    Validates *_transcription_translation.json files only if matching audio exists.
    Logs:
      - failed transcripts -> failed_transcripts_<ver>.txt
      - missing transcripts -> missing_transcripts_<ver>.txt
    """
    path = "pre-processing/Phase_1/"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    
    failed_log_path = os.path.join(path,f"failed_transcripts___{ver}.txt")
    missing_log_path = os.path.join(path,f"missing_transcripts__{ver}.txt") 
    ok_count, fail_count, skipped, missing = 0, 0, 0, 0

    with open(failed_log_path, "w", encoding="utf-8") as flog, \
         open(missing_log_path, "w", encoding="utf-8") as mlog:

        flog.write("---- Transcript Validation ----\n")
        mlog.write("---- Missing Transcript Files ----\n")

        batch_dirs = sorted([d for d in glob.glob(os.path.join(parent_dir, "urdu_03")) if os.path.isdir(d)])
        print(f" this is my batch dir ------ {batch_dirs}")
        for batch in batch_dirs:
            flog.write(f"## {os.path.basename(batch)}\n")
            mlog.write(f"## {os.path.basename(batch)}\n")

            # group by prefix
            folders = os.listdir(batch)
            groups = defaultdict(dict)
            for f in folders:
                fpath = os.path.join(batch, f)
                if not os.path.isdir(fpath):
                    continue
                if f.endswith("_audios"):
                    prefix = f[:-7]
                    groups[prefix]["audio"] = fpath
                elif any(x in f for x in ["transcribe", "translate", "transcription", "translation"]):
                    prefix = re.sub(r"_(transcribe|translate|transcription|translation).*", "", f)
                    groups[prefix]["trans"] = fpath

            # process each prefix
            for prefix, paths in groups.items():
                if "audio" not in paths:
                    continue

                audio_dir = paths["audio"]
                trans_dir = paths.get("trans")

                audio_files = sorted(glob.glob(os.path.join(audio_dir, "*.wav")), key=os.path.basename)
                transcript_files = sorted(
                    glob.glob(os.path.join(trans_dir, "*_transcription_translation.*")) if trans_dir else [],
                    key=lambda f: extract_key(os.path.basename(f))
                )

                # map transcripts by ID
                transcript_map = {extract_key(os.path.basename(f)): f for f in transcript_files}

                for a in audio_files:
                    aid = extract_key(os.path.basename(a))
                    tpath = transcript_map.get(aid)

                    if not tpath:
                        missing += 1
                        print(f"⚠️ Missing transcript for: {a}")
                        mlog.write(f"{a} | No transcript found\n")
                        continue

                    try:
                        ok, msg = validate_json_caption_file(tpath)
                        if ok:
                            ok_count += 1
                            print(f"✅ Valid: {tpath}")
                        else:
                            raise ValueError(msg)
                    except Exception as e:
                        fail_count += 1
                        print(f"❌ Failed: {tpath}")
                        flog.write(f"{tpath} | Error: {str(e)}\n")

                flog.write("---------------\n")
                mlog.write("---------------\n")

    print(f"\n📊 Done. {ok_count} valid, {fail_count} failed, {skipped} skipped, {missing} missing. Logs: {failed_log_path}, {missing_log_path}")
    return failed_log_path, missing_log_path



In [ ]:
parent = "/home/orgpu/dataset/URDU_03"
validate_transcripts_only(parent, ver="v0.1")

 this is my batch dir ------ ['/home/orgpu/dataset/URDU_03/urdu_03']
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/loose_talk_02_transcriptions/loose_talk_02_000026_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/loose_talk_02_transcriptions/loose_talk_02_000026_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/loose_talk_02_transcriptions/loose_talk_02_000026_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/loose_talk_02_transcriptions/loose_talk_02_000026_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/loose_talk_02_transcriptions/loose_talk_02_000026_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/loose_talk_02_transcriptions/loose_talk_02_000026_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/loose_talk_02_transcriptions/loose_talk_02_000026_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/loose_talk_

✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/happy_pakistan_transcriptions/happy_pakistan_000009_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/happy_pakistan_transcriptions/happy_pakistan_000010_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/happy_pakistan_transcriptions/happy_pakistan_000011_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/happy_pakistan_transcriptions/happy_pakistan_000012_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/happy_pakistan_transcriptions/happy_pakistan_000013_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/happy_pakistan_transcriptions/happy_pakistan_000014_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/happy_pakistan_transcriptions/happy_pakistan_000015_transcription_translation.txt
✅ Valid: /home/orgpu/dataset/URDU_03/urdu_03/happy_pakistan_transcriptions/happy_pakistan_000016_transcription_

('pre-processing/Phase_1/failed_transcripts___v0.1.txt',
 'pre-processing/Phase_1/missing_transcripts__v0.1.txt')

: 

In [ ]:
#WORKS BEST BUT COPIES

# do not use it

import re
import os
import shutil

BASE_PATH = "urdu2-pre-processing/Phase_2"
LOG_FILE = os.path.join(BASE_PATH, "fix_log.log")
FIXED_TS_LOG = os.path.join(BASE_PATH, "fixed_timestamps.log")
TIME_DILATION_LOG = os.path.join(BASE_PATH, "time_dilation.log")
TIME_DILATION_DIR = os.path.join(BASE_PATH, "time_dilation")


def fix_time(t: str, file_name: str, line_no: int) -> str:
    """
    Normalize timestamps into HH:MM:SS.mmm format and log changes with file name + line number.
    Detects rollover cases (e.g. 86s → 01:26, 60s → 01:00) and logs them separately.
    """
    t_original = t.strip()
    fixed = t_original
    dilation_fix = False

    try:
        t_clean = t_original.replace(",", ".")
        parts = t_clean.split(":")

        # Case 1: HH:MM:SS:MS
        if len(parts) == 4:
            h, m, s, ms = parts
            ms = ms.ljust(3, "0")[:3]
            fixed = f"{int(h):02d}:{int(m):02d}:{int(s):02d}.{ms}"

        # Case 2: HH:MM:SS or MM:SS:MS
        elif len(parts) == 3:
            h, m, s = parts

            # If the last part looks like milliseconds (2–3 digits, no dot) → treat as MM:SS:MS
            if (s.isdigit() and (len(s) == 2 or len(s) == 3)):
                mm, ss, ms = int(h), int(m), s.ljust(3, "0")[:3]
                fixed = f"00:{mm:02d}:{ss:02d}.{ms}"

            else:
                if "." in s:
                    sec, ms = s.split(".")
                    sec = int(sec)
                    ms = ms.ljust(3, "0")[:3]
                else:
                    sec, ms = int(s), "000"

                m = int(m)
                h = int(h)

                # Detect rollover corrections
                if sec >= 60:
                    add_m, sec = divmod(sec, 60)
                    m += add_m
                    dilation_fix = True
                if m >= 60:
                    add_h, m = divmod(m, 60)
                    h += add_h
                    dilation_fix = True

                fixed = f"{h:02d}:{m:02d}:{sec:02d}.{ms}"

        # Case 3: MM:SS(.ms)
        elif len(parts) == 2:
            m, s = parts
            if "." in s:
                sec, ms = s.split(".")
                ms = ms.ljust(3, "0")[:3]
            else:
                sec, ms = s, "000"

            fixed = f"00:{int(m):02d}:{int(sec):02d}.{ms}"  # force hours
            if int(sec) >= 60:
                dilation_fix = True

        # Case 4: Just seconds
        elif len(parts) == 1:
            sec = int(parts[0])
            fixed = f"00:00:{sec:02d}.000"
            if sec >= 60:
                dilation_fix = True

    except Exception:
        fixed = t_original

    if fixed != t_original:
        if dilation_fix:
            with open(TIME_DILATION_LOG, "a", encoding="utf-8") as logf:
                logf.write(f"{file_name} | line {line_no} | {t_original}  →  {fixed}\n")
        else:
            with open(FIXED_TS_LOG, "a", encoding="utf-8") as logf:
                logf.write(f"{file_name} | line {line_no} | {t_original}  →  {fixed}\n")

    return fixed, dilation_fix


def fix_timestamps_in_file(file_path):
    file_name = os.path.basename(file_path)
    dilation_detected = False

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    fixed_lines = []
    for i, line in enumerate(lines, start=1):  # track line numbers
        fixed_line = line
        matches = re.findall(r"\d{1,2}:\d{1,2}(?::\d{1,3})?(?:[.,:]\d+)?", line)
        for match in matches:
            fixed, dilation_fix = fix_time(match, file_name, i)
            fixed_line = fixed_line.replace(match, fixed)
            if dilation_fix:
                dilation_detected = True
        fixed_lines.append(fixed_line)

    with open(file_path, "w", encoding="utf-8") as f:
        f.writelines(fixed_lines)

    # copy to time_dilation folder if detected
    if dilation_detected:
        os.makedirs(TIME_DILATION_DIR, exist_ok=True)
        shutil.copy(file_path, os.path.join(TIME_DILATION_DIR, file_name))


def copy_problematic_transcriptions(log_file, out_ts="problematic_transcriptions", out_other="other_errors"):
    out_ts = os.path.join(BASE_PATH, out_ts)
    out_other = os.path.join(BASE_PATH, out_other)

    os.makedirs(out_ts, exist_ok=True)
    os.makedirs(out_other, exist_ok=True)

    ts_files, other_files = [], []

    with open(log_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if line.startswith("-"):
            continue

        parts = line.split("|")
        if len(parts) < 1:
            continue

        txt_path = parts[0].strip()
        error_msg = line.strip()

        if not os.path.exists(txt_path):
            continue

        if (
            "does not match format" in error_msg
            or "field 'start_time' invalid" in error_msg
            or "field 'end_time' invalid" in error_msg
        ):
            dest_path = os.path.join(out_ts, os.path.basename(txt_path))
            shutil.copy(txt_path, dest_path)
            ts_files.append(dest_path)
            with open(LOG_FILE, "a", encoding="utf-8") as logf:
                logf.write(f"{error_msg}\n")
        else:
            dest_path = os.path.join(out_other, os.path.basename(txt_path))
            shutil.copy(txt_path, dest_path)
            other_files.append(dest_path)
            with open(LOG_FILE, "a", encoding="utf-8") as logf:
                logf.write(f"{error_msg}\n")

    return ts_files, other_files


def fix_all_problematic(log_file, out_ts="problematic_transcriptions", out_other="other_errors"):
    os.makedirs(BASE_PATH, exist_ok=True)
    open(LOG_FILE, "w").close()
    open(FIXED_TS_LOG, "w").close()
    open(TIME_DILATION_LOG, "w").close()

    ts_files, other_files = copy_problematic_transcriptions(log_file, out_ts, out_other)

    count_fixed = 0
    for f in ts_files:
        fix_timestamps_in_file(f)
        count_fixed += 1
        print(f"🛠️ Fixed timestamp: {f}")

    for f in other_files:
        print(f"⚠️ Moved (manual check needed): {f}")

    total_files = len(ts_files) + len(other_files)

    with open(LOG_FILE, "a", encoding="utf-8") as logf:
        logf.write(f"\n✅ Fixed {count_fixed} / {total_files} problematic transcription files\n")

    print(f"\n✅ Timestamp fixes logged in {LOG_FILE}")
    print(f"📂 Fixed timestamps logged in {FIXED_TS_LOG}")
    print(f"📂 Time dilation fixes logged in {TIME_DILATION_LOG}")
    print(f"📂 Time dilation files copied to: {TIME_DILATION_DIR}")
    print(f"📂 Other error files copied to: {os.path.join(BASE_PATH, out_other)}")
    
    
    
fix_all_problematic("/home/orgpu/copied/voicepro_training/ft-whisper/urdu2-pre-processing/Phase_1/failed_urdu_02_transcripts_v0.1.txt")    

In [ ]:
fix_all_problematic("/home/orgpu/copied/voicepro_training/ft-whisper/Dataset_v02/raw/Punjabi_Batch_01")

# **Phase_2:** Fixing Timestamp Issues


In [18]:
# FIXES ORIGINAL FILES - sssss this sssss

import re
import os
import shutil

import json


BASE_PATH = "pre-processing/Phase_2"
LOG_FILE = os.path.join(BASE_PATH, "fix_log.log")
FIXED_TS_LOG = os.path.join(BASE_PATH, "fixed_timestamps.log")
TIME_DILATION_LOG = os.path.join(BASE_PATH, "time_dilation.log")
TIME_DILATION_DIR = os.path.join(BASE_PATH, "time_dilation")


def fix_time(t: str, file_name: str, line_no: int) -> str:
    """
    Normalize timestamps into HH:MM:SS.mmm format and log changes with file name + line number.
    Detects rollover cases (e.g. 86s → 01:26, 60s → 01:00) and logs them separately.
    """
    t_original = t.strip()
    fixed = t_original
    dilation_fix = False

    try:
        t_clean = t_original.replace(",", ".")
        parts = t_clean.split(":")

        # Case 1: HH:MM:SS:MS
        if len(parts) == 4:
            h, m, s, ms = parts
            ms = ms.ljust(3, "0")[:3]
            fixed = f"{int(h):02d}:{int(m):02d}:{int(s):02d}.{ms}"

        # Case 2: HH:MM:SS or MM:SS:MS
        elif len(parts) == 3:
            h, m, s = parts

            # If the last part looks like milliseconds (2–3 digits, no dot) → treat as MM:SS:MS
            if (s.isdigit() and (len(s) == 2 or len(s) == 3)):
                mm, ss, ms = int(h), int(m), s.ljust(3, "0")[:3]
                fixed = f"00:{mm:02d}:{ss:02d}.{ms}"

            else:
                if "." in s:
                    sec, ms = s.split(".")
                    sec = int(sec)
                    ms = ms.ljust(3, "0")[:3]
                else:
                    sec, ms = int(s), "000"

                m = int(m)
                h = int(h)

                # Detect rollover corrections
                if sec >= 60:
                    add_m, sec = divmod(sec, 60)
                    m += add_m
                    dilation_fix = True
                if m >= 60:
                    add_h, m = divmod(m, 60)
                    h += add_h
                    dilation_fix = True

                fixed = f"{h:02d}:{m:02d}:{sec:02d}.{ms}"

        # Case 3: MM:SS(.ms)
        elif len(parts) == 2:
            m, s = parts
            if "." in s:
                sec, ms = s.split(".")
                ms = ms.ljust(3, "0")[:3]
            else:
                sec, ms = s, "000"

            fixed = f"00:{int(m):02d}:{int(sec):02d}.{ms}"  # force hours
            if int(sec) >= 60:
                dilation_fix = True

        # Case 4: Just seconds
        elif len(parts) == 1:
            sec = int(parts[0])
            fixed = f"00:00:{sec:02d}.000"
            if sec >= 60:
                dilation_fix = True

    except Exception:
        fixed = t_original

    if fixed != t_original:
        if dilation_fix:
            with open(TIME_DILATION_LOG, "a", encoding="utf-8") as logf:
                logf.write(f"{file_name} | line {line_no} | {t_original}  →  {fixed}\n")
        else:
            with open(FIXED_TS_LOG, "a", encoding="utf-8") as logf:
                logf.write(f"{file_name} | line {line_no} | {t_original}  →  {fixed}\n")

    return fixed, dilation_fix


def fix_timestamps_in_file(file_path):
    file_name = os.path.basename(file_path)
    dilation_detected = False

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    fixed_lines = []
    for i, line in enumerate(lines, start=1):  # track line numbers
        fixed_line = line
        matches = re.findall(r"\d{1,2}:\d{1,2}(?::\d{1,3})?(?:[.,:]\d+)?", line)
        for match in matches:
            fixed, dilation_fix = fix_time(match, file_name, i)
            fixed_line = fixed_line.replace(match, fixed)
            if dilation_fix:
                dilation_detected = True
        fixed_lines.append(fixed_line)

    with open(file_path, "w", encoding="utf-8") as f:
        f.writelines(fixed_lines)

    # copy to time_dilation folder if detected
    if dilation_detected:
        os.makedirs(TIME_DILATION_DIR, exist_ok=True)
        shutil.copy(file_path, os.path.join(TIME_DILATION_DIR, file_name))

######################### Language key block here ################################

# def fix_language_key(file_path):
#     with open(file_path, "r", encoding="utf-8", errors="replace") as f:
#         text = f.read()

#     # Only replace if "urdu" key exists
#     if '"urdu"' in text:
#         new_text = text.replace('"urdu"', '"pashto"')
#         with open(file_path, "w", encoding="utf-8") as f:
#             f.write(new_text)
#         print(f"✅ Urdu→Pashto fixed in: {os.path.basename(file_path)}")
#         return True
#     return False




def get_problematic_files(log_file):
    """Read log and return list of files that need fixing (original paths)."""
    files = []

    with open(log_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line or line.startswith("-"):
            continue

        parts = line.split("|")
        if not parts:
            continue

        txt_path = parts[0].strip()
        error_msg = line.strip()

        if not os.path.exists(txt_path):
            continue

        # Only flag timestamp-related errors
        if (
            "does not match format" in error_msg
            or "field 'start_time' invalid" in error_msg
            or "field 'end_time' invalid" in error_msg
        ):
            files.append(txt_path)

    return files


def fix_all_problematic(log_file):
    os.makedirs(BASE_PATH, exist_ok=True)
    open(LOG_FILE, "w").close()
    open(FIXED_TS_LOG, "w").close()
    open(TIME_DILATION_LOG, "w").close()

    ts_files = get_problematic_files(log_file)

    count_fixed = 0
    

    for f in ts_files:

        fix_timestamps_in_file(f)   # ✅ Fix original file in place
        count_fixed += 1
        print(f"🛠️ Fixed timestamp: {f}")

        

    total_files = len(ts_files)

    with open(LOG_FILE, "a", encoding="utf-8") as logf:
        logf.write(f"\n✅ Fixed {count_fixed} / {total_files} problematic transcription files\n")

    
    print(f" Total files with timestamp fixed: {count_fixed} ")
    print(f"\n✅ Timestamp fixes logged in {LOG_FILE}")
    print(f"📂 Fixed timestamps logged in {FIXED_TS_LOG}")
    print(f"📂 Time dilation fixes logged in {TIME_DILATION_LOG}")
    print(f"📂 Time dilation files copied to: {TIME_DILATION_DIR}")


In [19]:
fix_all_problematic("/home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/Phase_1/failed_transcripts___v0.1.txt")

🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000001_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000002_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000003_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000004_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000005_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000006_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000007_transcrip

🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000040_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000041_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000042_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000043_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000044_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000045_transcription_translation.json
🛠️ Fixed timestamp: /home/orgpu/dataset/URDU_03/urdu_03/adnan_family_vlogs_transcriptions/adnan_family_vlog_000047_transcrip

In [ ]:
############# test block for language ###############

file_path = "/home/orgpu/dataset/pashto_dataset/Batch_pashto/Batch_pashto_01/ghani_khan_transcriptions/ghani_khan_000222_transcription_translation.txt"

with open(file_path, "r", encoding="utf-8", errors="replace") as f:
    text = f.read()

print("Does it contain 'urdu'?", "urdu" in text)
print("Does it contain '\"urdu\"'?", '"urdu"' in text)
print("Sample around it:")
import re
match = re.search(r".{0,40}urdu.{0,40}", text)
if match:
    print(match.group())
else:
    print("No visible 'urdu' found in text snippet.")


# **Phase_3:** Fixing Time Anomalies

In [1]:
import re
import json
import os
import shutil
from collections import defaultdict


BASE_PATH = "pre-processing/Phase_3"
ANOMALY_FILES_DIR = os.path.join(BASE_PATH, "anomaly_files")
FIXED_FILES_DIR = os.path.join(BASE_PATH, "fixed")
ANOMALY_LOG = os.path.join(BASE_PATH, "anomaly_log.log")


def load_clean_json(file_path):
    """Load JSON safely, stripping ``` fences if present."""
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    content = re.sub(r"^```(?:json)?\s*", "", content)
    content = re.sub(r"```$", "", content)
    content = re.sub(r",\s*([\]}])", r"\1", content)
    return json.loads(content)


def parse_time(t: str) -> float:
    """Convert HH:MM:SS.mmm or MM:SS.mmm into total seconds."""
    t = t.strip().replace(",", ".").replace("_", ":")
    parts = t.split(":")
    if len(parts) == 3:  # HH:MM:SS
        h, m, s = parts
        if "." in s:
            sec, ms = s.split(".")
            return int(h) * 3600 + int(m) * 60 + int(sec) + int(ms) / 1000
        return int(h) * 3600 + int(m) * 60 + int(s)
    elif len(parts) == 2:  # MM:SS
        m, s = parts
        if "." in s:
            sec, ms = s.split(".")
            return int(m) * 60 + int(sec) + int(ms) / 1000
        return int(m) * 60 + int(s)
    raise ValueError(f"Bad time format: {t}")


def format_time(seconds: float) -> str:
    """Convert seconds back to HH:MM:SS.mmm format."""
    if seconds < 0:
        seconds = 0.0
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int(round((seconds - int(seconds)) * 1000))
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


def fix_time_anomalies(file_path, root_folder):
    """Fix anomalies in one file. Returns (anomalies list, fixed_path)."""
    data = load_clean_json(file_path)
    anomalies = []

    M_SHIFT = 60.0
    M_DETECT_MIN, M_DETECT_MAX = 55.0, 65.0

    prev_end = None
    for i, seg in enumerate(data):
        try:
            start = parse_time(seg["start_time"])
            end = parse_time(seg["end_time"])
        except Exception:
            continue

        # ~1m mistake within segment
        diff = end - start
        if M_DETECT_MIN < diff < M_DETECT_MAX:
            anomalies.append((i, seg["end_time"], "within-seg 1m jump"))
            end -= M_SHIFT
            seg["end_time"] = format_time(end)

        # ~1m mistake across segments
        if prev_end is not None:
            gap = start - prev_end
            if M_DETECT_MIN < gap < M_DETECT_MAX:
                anomalies.append((i, seg["start_time"], "cross-seg 1m shift"))
                start -= M_SHIFT
                end -= M_SHIFT
                seg["start_time"] = format_time(start)
                seg["end_time"] = format_time(end)

        # Fix invalid order
        if end <= start:
            anomalies.append((i, seg.get("end_time", ""), "end <= start"))
            end = start + 0.5
            seg["end_time"] = format_time(end)

        if prev_end is not None and start < prev_end:
            anomalies.append((i, seg["start_time"], "overlap"))
            start = prev_end + 0.01
            if end <= start:
                end = start + 0.5
            seg["start_time"] = format_time(start)
            seg["end_time"] = format_time(end)

        prev_end = end

    # Save fixed file under FIXED_FILES_DIR (mirroring folder structure)
    rel_path = os.path.relpath(file_path, root_folder)
    fixed_path = os.path.join(root_folder, FIXED_FILES_DIR, rel_path)
    os.makedirs(os.path.dirname(fixed_path), exist_ok=True)

    with open(fixed_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return anomalies, fixed_path


def check_folder(root_folder):
    """Check all files in a folder, save fixed ones in /fixed, copy anomalous originals in /anomaly_files, log neatly."""
    os.makedirs(ANOMALY_FILES_DIR, exist_ok=True)
    os.makedirs(os.path.join(root_folder, FIXED_FILES_DIR), exist_ok=True)

    anomaly_files_set = set()
    all_anomalies = defaultdict(lambda: defaultdict(list))  # batch → file → anomalies

    for root, _, files in os.walk(root_folder):
        for name in files:
            if not (name.lower().endswith(".json") or name.lower().endswith(".txt")):
                continue
            file_path = os.path.join(root, name)

            try:
                anomalies, fixed_path = fix_time_anomalies(file_path, root_folder)
                if anomalies:
                    anomaly_files_set.add(file_path)
                    shutil.copy(file_path, os.path.join(ANOMALY_FILES_DIR, name))

                    # detect batch (e.g., Batch_1 from path)
                    batch_name = next((part for part in file_path.split(os.sep) if part.startswith("pashto_")), "Other")
                    all_anomalies[batch_name][name].extend(anomalies)

                print(f"✅ {file_path} → {fixed_path} ({len(anomalies)} anomalies fixed)")
            except Exception as e:
                print(f"❌ Failed {file_path}: {e}")

    # Write sorted anomaly log
    with open(ANOMALY_LOG, "w", encoding="utf-8") as logf:
        for batch in sorted(all_anomalies.keys(), key=lambda b: int(re.search(r"\d+", b).group()) if re.search(r"\d+", b) else 9999):
            logf.write(f"## {batch}\n")
            for fname in sorted(all_anomalies[batch].keys()):
                for i, t, msg in sorted(all_anomalies[batch][fname], key=lambda x: x[0]):  # sort by segment index
                    logf.write(f"{fname} | segment {i} | time={t} | {msg}\n")
            logf.write("---------------\n")

    print(f"\n📂 Total files with anomalies: {len(anomaly_files_set)} (saved in {ANOMALY_FILES_DIR})")
    print(f"📝 Anomaly log written to {ANOMALY_LOG}")
    print(f"📂 Fixed files saved under {os.path.join(root_folder, FIXED_FILES_DIR)}")


In [2]:
check_folder("/home/orgpu/dataset/PASHTO_03/pashto_03")

❌ Failed /home/orgpu/dataset/PASHTO_03/pashto_03/parachinar_press_audios/durations.txt: Expecting value: line 1 column 1 (char 0)
❌ Failed /home/orgpu/dataset/PASHTO_03/pashto_03/ghani_khan_audios/durations_2.txt: Expecting value: line 1 column 1 (char 0)
✅ /home/orgpu/dataset/PASHTO_03/pashto_03/pashto_yousaf_tiktok_transcriptions/pashto_yousaf_tiktok_000329_transcription_translation.json → /home/orgpu/dataset/PASHTO_03/pashto_03/pre-processing/Phase_3/fixed/pashto_yousaf_tiktok_transcriptions/pashto_yousaf_tiktok_000329_transcription_translation.json (0 anomalies fixed)
✅ /home/orgpu/dataset/PASHTO_03/pashto_03/pashto_yousaf_tiktok_transcriptions/pashto_yousaf_tiktok_000319_transcription_translation.json → /home/orgpu/dataset/PASHTO_03/pashto_03/pre-processing/Phase_3/fixed/pashto_yousaf_tiktok_transcriptions/pashto_yousaf_tiktok_000319_transcription_translation.json (0 anomalies fixed)
✅ /home/orgpu/dataset/PASHTO_03/pashto_03/pashto_yousaf_tiktok_transcriptions/pashto_yousaf_tiktok

EXCLUDING ALL PROBLEMATIC FILES TO A NEW FOLDER

In [13]:
import os
import shutil

# ==== CONFIG ====
FAILED_TXT = "/home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/Phase_1/failed_transcripts___v0.1.txt"    # TXT with JSON parse errors / missing fields
MISSING_TXT = "/home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/Phase_1/missing_transcripts__v0.1.txt"  # TXT with missing transcript info
PROBLEM_FOLDER = "/home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/PROBLEM_FILES"     # where to move problematic files

# ==== CREATE PROBLEMATIC FOLDER ====
os.makedirs(PROBLEM_FOLDER, exist_ok=True)

def move_file(file_path, problem_folder):
    """Move a file to problem_folder, preserving filename."""
    if os.path.exists(file_path):
        try:
            dest_path = os.path.join(problem_folder, os.path.basename(file_path))
            shutil.move(file_path, dest_path)
            print(f"Moved: {file_path} → {dest_path}")
        except Exception as e:
            print(f"Failed to move {file_path}: {e}")
    else:
        print(f"File not found (skipping): {file_path}")

def parse_and_move(txt_file, problem_folder):
    """Read a TXT log, extract file paths, move them to problem folder."""
    with open(txt_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        # Skip empty lines, headers, or separator lines
        if not line or line.startswith("----") or line.startswith("##") or line.startswith("-"):
            continue
        # Extract the file path before the first " | " (pipe)
        if "|" in line:
            file_path = line.split("|")[0].strip()
            move_file(file_path, problem_folder)

# ==== MOVE FILES FROM BOTH LOGS ====
parse_and_move(FAILED_TXT, PROBLEM_FOLDER)
parse_and_move(MISSING_TXT, PROBLEM_FOLDER)

print(f"\nAll problematic files moved to: {PROBLEM_FOLDER}")


Moved: /home/orgpu/dataset/PASHTO_03/pashto_03/ghani_khan_audios/ghani_khan_000005.wav → /home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/PROBLEM_FILES/ghani_khan_000005.wav
Moved: /home/orgpu/dataset/PASHTO_03/pashto_03/ghani_khan_audios/ghani_khan_000012.wav → /home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/PROBLEM_FILES/ghani_khan_000012.wav
Moved: /home/orgpu/dataset/PASHTO_03/pashto_03/ghani_khan_audios/ghani_khan_000032.wav → /home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/PROBLEM_FILES/ghani_khan_000032.wav
Moved: /home/orgpu/dataset/PASHTO_03/pashto_03/ghani_khan_audios/ghani_khan_000034.wav → /home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/PROBLEM_FILES/ghani_khan_000034.wav
Moved: /home/orgpu/dataset/PASHTO_03/pashto_03/ghani_khan_audios/ghani_khan_000053.wav → /home/orgpu/copied/voicepro_training/ft-whisper/pre-processing/PROBLEM_FILES/ghani_khan_000053.wav
Moved: /home/orgpu/dataset/PASHTO_03/pashto_03/ghani_khan_au